# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/engyelgamal18/flyrank-ml-internship-engy/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule prioritizes pages that are still visible in search but have weaker than expected CTR for their search position. A page receives a higher score when it has meaningful impressions and relatively low CTR. The reason code is VISIBLE_LOW_CTR and the ation label is CTR_REVIEW.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)

df = pd.read_parquet(march_path)

print("Rows loaded:",len(df))

Rows loaded: 9841378


In [ ]:
import numpy as np
import os

queue = df[
    (df["gsc_data_available"] == True) &
    (df["gsc_impressions"] > 0)
][
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
].copy()

queue["ctr"] = queue["gsc_clicks"] / queue["gsc_impressions"]

queue["expected_ctr"] = np.select(
    [
        queue["gsc_avg_position"] <= 3,
        queue["gsc_avg_position"] <= 10,
        queue["gsc_avg_position"] <= 20
    ],
    [0.004576, 0.003473, 0.002770],
    default=0.001289
)

queue["score"] = (
    queue["gsc_impressions"]*
    (queue["expected_ctr"] - queue["ctr"]).clip(lower=0)
)


queue["reason_code"] = "VISIBLE_LOW_CTR"
queue["action"] = "CTR_REVIEW"

queue = queue.sort_values("score", ascending=False)

queue.head(10)

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("csv written: work/outputs/baseline_action_score.csv")

csv written: work/outputs/baseline_action_score.csv


In [ ]:
impression_buckets = pd.cut(
    df["gsc_impressions"],
    bins=[-1, 0, 10, 100, 500, float("inf")],
    labels=["0", "1-10", "11-100", "101-500", "500+"]
)

signall =(
    df.assign(impression_bucket=impression_buckets)
    .groupby("impression_bucket", observed=True)
    .size()
    .reset_index(name="n")
)

signall

Visibility verdict: CONFIRMED. The data contains a meaningful group of pages with substantial search visibility including 101,136 rows with more than 500 impressions. This supports using impressions as part of the baseline rule to identify stale pages that are still visible.

In [ ]:
ctr_check = (
    df[df["gsc_data_available"] == True]
    .assign(
        ctr=lambda x: x["gsc_clicks"] / x["gsc_impressions"].replace(0, pd.NA)
    )
    .dropna(subset=["ctr", "gsc_avg_position"])
)

ctr_check["position_bucket"] = pd.cut(
    ctr_check["gsc_avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    include_lowest=True
)

signal2 = (
    ctr_check.groupby("position_bucket", observed=True)
    .agg(
        n=("ctr", "size"),
        avg_ctr=("ctr", "mean")
    )
    .reset_index()
)

signal2

CTR-vs-position verdict: CONFIRMED. Average CTR decreased as search position gets worse from about 0.476% in the top position bucket to about 0.129% beyond position 20. this supports the CTR-vs-position logic used behind CTR fix decisions.

In [ ]:
[c for c in df.columns if "update" in c.lower() or "age" in c.lower() or "stale" in c.lower()]

In [ ]:
print(df.columns.tolist())

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top 10 review

In [ ]:
top10 = queue.head(10)[
    [
        "report_date",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "expected_ctr",
        "score",
        "reason_code",
        "action"
    ]
].copy()

top10

1.CTR_REVIEW__40,084 impressions with only 1 click and CTR  far below expected. Confidence: high. Wrong if the query is naturally zero click or tracking is incomplete
2.CTR_REVIEW__39,003 impressions with 2 clicks despite an average position around 2.76. Confidence: high. Wrong if search intent is statisfied directly on the result page.
3.CTR_REVIEW__33,383 impressions with 0 clicks while ranking very highly. Confidence: high. Wrong if impressions or clicks are being measures incorrectly.
4.CTR_REVIEW__32,958 impressions with 0 clicks and a very strong average position. Confidence: high. Wrong if this is a zero click search result.
5.CTR_REVIEW__32,765 impressions with only 2 clicks and CTR much lower than expected. Confidence: high. Wrong if the query does not normally require a website click.
6.CTR_REVIEW__31,472 impressions with 0 clicks despite top search visibility. Confidence:high. Wrong if tracking is missing or incomplete.
7.CTR_REVIEW__30,964 impression with only 1 click at a very strong position. Confidence: high. Wrong if the result already answers the query directly in search.
8.CTR_REVIEW__ 30,573 impressions with  only 2 clicks and CTR well below the expected level. Confidence: high. Wrong if the search intent is informational and mostly zero click.
9.CTR_REVIEW__30,573 impressions with 2 clicks despite ranking near the top. Confidence: high. Wrong if the page is appearing for irrelevant queries.
10.CTR_REVIEW__28,973 impressions with 0 clicks while ranking extremely highly. Confidence: high. Wrong if there is a measurement issue or the SERP itself satisfies the user.



## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Some high ranked pages may be false positive if the search result already answers the user's question without a click , if the page appears for irrelevant queries or if click tracking is incomplete. These cases would make a low CTR look wrose than it really is. Leakage check: The rule only uses current month Search Console signals available at the decision moment: impressions, clicks and average position. It does not use future window data, label derived columns or existing product flags.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.